In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

# 1. Tải và tiền xử lý dữ liệu Pima Indians Diabetes Dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
           'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

# Tách features (X) và target (y)
X = df.drop('Outcome', axis=1).values
y = df['Outcome'].values

# Chia tập dữ liệu train / test (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Chuẩn hóa dữ liệu (Standardization)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Chuyển đổi sang PyTorch Tensor
X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32).unsqueeze(1)

# 2. Định nghĩa kiến trúc Mạng Nơ-ron (MLP Binary Classifier)
class DiabetesNN(nn.Module):
    def __init__(self, input_dim):
        super(DiabetesNN, self).__init__()
        self.fc1 = nn.Linear(input_dim, 16)
        self.relu1 = nn.ReLU()
        self.fc2 = nn.Linear(16, 8)
        self.relu2 = nn.ReLU()
        self.fc3 = nn.Linear(8, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.relu1(self.fc1(x))
        x = self.relu2(self.fc2(x))
        x = self.sigmoid(self.fc3(x))
        return x

model = DiabetesNN(input_dim=X_train.shape[1])

# 3. Định nghĩa Hàm mất mát và Thuật toán tối ưu
criterion = nn.BCELoss()  # Binary Cross-Entropy Loss
optimizer = optim.Adam(model.parameters(), lr=0.005)

# 4. Huấn luyện mô hình (Training Loop)
epochs = 150
print("--- Đang huấn luyện mô hình ---")
for epoch in range(epochs):
    model.train()

    # Forward pass
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    # Backward pass & Optimize
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch [{epoch+1}/{epochs}], Loss: {loss.item():.4f}")

# 5. Đánh giá mô hình trên tập kiểm tra
model.eval()
with torch.no_grad():
    predictions = model(X_test_tensor)
    predicted_classes = (predictions >= 0.5).float()

    acc = accuracy_score(y_test_tensor, predicted_classes)
    print(f"\nĐộ chính xác (Accuracy) trên tập test: {acc * 100:.2f}%\n")
    print("Báo cáo chi tiết (Classification Report):")
    print(classification_report(y_test_tensor, predicted_classes, target_names=['Không bệnh', 'Mắc bệnh']))

--- Đang huấn luyện mô hình ---
Epoch [30/150], Loss: 0.5138
Epoch [60/150], Loss: 0.4394
Epoch [90/150], Loss: 0.4165
Epoch [120/150], Loss: 0.3955
Epoch [150/150], Loss: 0.3689

Độ chính xác (Accuracy) trên tập test: 73.38%

Báo cáo chi tiết (Classification Report):
              precision    recall  f1-score   support

  Không bệnh       0.79      0.80      0.80       100
    Mắc bệnh       0.62      0.61      0.62        54

    accuracy                           0.73       154
   macro avg       0.71      0.71      0.71       154
weighted avg       0.73      0.73      0.73       154





Dựa trên kết quả huấn luyện và đánh giá mô hình Multi-Layer Perceptron (MLP) trên tập dữ liệu **Pima Indians Diabetes**, chúng ta có các nhận xét chi tiết sau:

#### 1. Quá trình huấn luyện (Training Process)
* **Giá trị Loss giảm đều**: Giá trị Loss (Hàm mất mát Binary Cross-Entropy) giảm liên tục từ `0.5138` ở epoch 30 xuống còn `0.3689` ở epoch 150. Điều này cho thấy thuật toán tối ưu Adam đang hoạt động tốt và mô hình đang học được các đặc trưng từ tập dữ liệu huấn luyện.

#### 2. Hiệu năng trên tập kiểm tra (Test Set Performance)
* **Độ chính xác toàn cục (Accuracy)**: Đạt **73.38%**. Đây là một kết quả ở mức khá ổn đối với một kiến trúc mạng nơ-ron cơ bản trên tập dữ liệu có kích thước tương đối nhỏ và bị mất cân bằng nhẹ này.

#### 3. Phân tích chi tiết báo cáo phân loại (Classification Report)
* **Lớp 0 (Không bệnh - 100 mẫu)**:
  * **Precision (Độ chính xác)**: **79%** (Trong số các ca mô hình dự đoán là không bệnh, có 79% thực sự không bệnh).
  * **Recall (Độ nhạy)**: **80%** (Mô hình tìm ra được 80% tổng số ca thực tế không bệnh).
  * **F1-score**: **0.80** — Hiệu năng dự đoán đối với nhóm không bệnh rất ổn định.

* **Lớp 1 (Mắc bệnh - 54 mẫu)**:
  * **Precision (Độ chính xác)**: **62%** (Trong số các ca dự đoán mắc bệnh, chỉ có 62% thực sự mắc bệnh, tỷ lệ dương tính giả còn hơi cao).
  * **Recall (Độ nhạy)**: **61%** (Mô hình chỉ phát hiện được 61% số người thực sự mắc bệnh, tỷ lệ bỏ sót/âm tính giả là khoảng 39%).
  * **F1-score**: **0.62** — Thấp hơn đáng kể so với nhóm không bệnh.

#### 4. Đánh giá chung & Hướng cải thiện
* **Vấn đề mất cân bằng dữ liệu**: Tập test có 100 mẫu không bệnh nhưng chỉ có 54 mẫu mắc bệnh (tỉ lệ gần 2:1). Điều này dẫn đến việc mô hình nhận diện nhóm không bệnh tốt hơn hẳn nhóm mắc bệnh.
* **Hướng cải thiện khả thi**:
  1. **Điều chỉnh ngưỡng phân loại (Classification Threshold)**: Thay vì mặc định là `0.5`, có thể hạ ngưỡng xuống một chút để tăng độ nhạy (Recall) cho nhóm mắc bệnh (vì trong y tế, việc bỏ sót người bệnh nguy hiểm hơn việc chẩn đoán nhầm).
  2. **Áp dụng các kỹ thuật cân bằng dữ liệu**: Sử dụng các phương pháp như SMOTE hoặc điều chỉnh trọng số lớp (`weight` trong `BCELoss` hoặc `BCEWithLogitsLoss`).
  3. **Thêm Dropout hoặc Weight Decay**: Để tránh overfitting và cải thiện khả năng tổng quát hóa của mô hình.